In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df_copy = pd.read_csv('Training_set.csv',delimiter=",")

In [ ]:
df = pd.read_csv('Training_set.csv',delimiter=",")
df.head()

In [ ]:
df_labels = pd.read_csv('Training_set_labels.csv',delimiter=",")
df_labels.head()

In [ ]:
df_labels['status_group'].value_counts(normalize=True)

In [ ]:
df_labels['status_group'].hist()

In [ ]:
#Checking column data types

print(df.info())

Data Observation
59400 rows and 39 columns are present in the  training dataset.
The population around the water pumps ranges from  0 to 30500.
Construction year has values as 0 and ranges from 1960 to 2013.
The training data contains many columns with identical values. These are classified and grouped into few values. For better data cleaning, these columns are deleted, allowing us to use higher-grouped columns with fewer values.
recorded_by column has one value repeated which can be dropped.


In [ ]:
# Analyzing the Data distribution 

#print(df.describe())

df.describe()

In [ ]:
# Check for missing values


df.isnull().sum().sort_values(ascending=False).head(20)

In [ ]:
# Convert numeric codes to categorical
for col in ['region_code', 'district_code']:
    df[col] = df[col].astype('category')

In [ ]:
# Ticket 1.2.1 — Handle missing values

#Handle Missing Values for the categorical columns

cat_cols = ['scheme_name','scheme_management' , 'installer','funder']
df[cat_cols] = df[cat_cols].fillna('Unknown')

In [ ]:
#Handle Missing Values for the Boolean columns

for col in ['public_meeting', 'permit']:
    mode_value = df[col].mode()[0]
    df[col] = df[col].fillna(mode_value)
    df[col] = df[col].infer_objects(copy=False)
    df[col] = df[col].astype(bool)

In [ ]:
#  Handling "0" as missing for geographical  columns 

geo_cols = ['longitude', 'latitude', 'gps_height']
df[geo_cols] = df[geo_cols].replace({0: np.nan}) 

In [ ]:
for col in geo_cols:
    # Compute region median safely
    region_medians = df.groupby('region')[col].median()

    # Fill with region median first
    df[col] = df.apply(
        lambda row: region_medians[row['region']]
        if pd.notna(region_medians[row['region']])
        else np.nan, axis=1
    )

    # Now fill any leftover NaN with the global median
    df[col] = df[col].fillna(df[col].median())

In [ ]:
df['gps_height'] = df['gps_height'].replace(0, np.nan)
df['wpt_name'] = df['wpt_name'].fillna('unknown')
df['subvillage'] = df['subvillage'].fillna('unknown')

In [ ]:
df['amount_tsh'] = df['amount_tsh'].replace(0, np.nan)
# Median imputation per region 
df['amount_tsh'] = df.groupby('region')['amount_tsh'].transform(lambda x: x.fillna(x.median()if not x.dropna().empty else df['amount_tsh'].median()))
# If still missing , fill with global median
df['amount_tsh'] = df['amount_tsh'].fillna(df['amount_tsh'].median())


df['construction_year'] = df['construction_year'].replace(0, np.nan)
# Impute using median by region 
df['construction_year'] = df.groupby('region')['construction_year'].transform(lambda x: x.fillna(x.median()if not x.dropna().empty else df['construction_year'].median()))
# If still missing , fill with global median
df['construction_year'] = df['construction_year'].fillna(df['construction_year'].median())

In [ ]:
# Drop columns 

drop_cols = [c for c in ['recorded_by','id','num_private'] if c in df.columns]
df = df.drop(columns=drop_cols)

print("Remaining NA counts:\n", df.isna().sum()[df.isna().sum()>0])

In [ ]:
train_data = pd.concat([df, df_labels.drop('id', axis=1)], axis=1)
train_data.head(2)

In [ ]:
# Visualizing whether water quality has some effect on functionality status of the water pumps

fig, ax  = plt.subplots(figsize=(20,8))
sns.histplot(data=train_data, x="water_quality", hue="status_group", multiple="dodge",palette='dark', shrink=.8, ax=ax)
ax.set_xlabel("Water Quality")
ax.set_ylabel("Number of water pumps");

#Obseravtion
#Soft water has more functional pumps than non-functional
#Salty water results in more non-functional pumps 

In [ ]:
# Visualizing  Region wise functionality status of the water pumps

fig, ax  = plt.subplots(figsize=(30,8))
sns.histplot(data=train_data, x="region", hue="status_group", multiple="dodge",palette='dark', shrink=.8, ax=ax)
ax.set_xlabel("Region")
ax.set_ylabel("Number of water pumps");

#Observation
#Mbeye ,Morogoro and Shinyanga regions have more non-functional pumps.They need more attention.
#Iringa region is doing well as in case of more functional pumps

In [ ]:
# Visualizing whether extraction type has some effect on functionality status of the water pumps

fig, ax  = plt.subplots(figsize=(20,8))
sns.histplot(data=train_data, x="extraction_type_class", hue="status_group", multiple="dodge", palette='dark', shrink=.8, ax=ax)
ax.set_xlabel("Extraction type")
ax.set_ylabel("Number of water pumps");

#Observation
#More non functional pumps are under Extraction type  "gravity" ,"handpump" and "other"

In [ ]:
# Visualizing whether source of water has some effect on functionality status of the water pumps

fig, ax  = plt.subplots(figsize=(20,8))
sns.histplot(data=train_data, x="source_type", hue="status_group", multiple="dodge",palette='dark', shrink=.8, ax=ax)
ax.set_xlabel("source")
ax.set_ylabel("Number of water pumps");

#Observation
# Functional pipes are working well when having the Spring source
# Non-Functional pipes are more in shallow well source when compared to other sources

In [ ]:
# Visualizing  Quantity of water  effect on functionality status of the water pumps

fig, ax  = plt.subplots(figsize=(20,8))
sns.histplot(data=train_data, x="quantity_group", hue="status_group", multiple="dodge",palette='dark', shrink=.8, ax=ax)
ax.set_xlabel("Quantity")
ax.set_ylabel("Number of water pumps");

#Observation
#Dry results in more non-functional pumps

In [ ]:
# Visualizing  payment type  effect on functionality status of the water pumps

fig, ax  = plt.subplots(figsize=(20,8))
sns.histplot(data=train_data, x="payment_type", hue="status_group", multiple="dodge",palette='dark', shrink=.8, ax=ax)
ax.set_xlabel("Payment type")
ax.set_ylabel("Number of water pumps");

# Observation
# Non functional pumps are are very high under Never pay category

In [ ]:
#Corerelation heat map for finding the 



f, ax = plt.subplots(figsize=(10, 8))
corr = df.corr(numeric_only=True)
sns.heatmap(corr,cmap='RdBu')

#Observation

#gps_height and and Construction_year are slightly correlated
#longitude and latitude shows strong negative correlation (geographical coordinates vary inversely across regions in Tanzania)

In [ ]:
#numeric skew data distribution


numeric_columns = df.select_dtypes(include='number').columns


df[numeric_columns].hist(bins=30, figsize=(16, 12), grid=False)
plt.suptitle("Histograms of Numeric Features", fontsize=16)
plt.tight_layout()
plt.show()

#Observation

In [ ]:
#Handling outliers in numeric data